# Benchmark: `qwen-engine` (W4A16) vs. `llama.cpp` (`Q4_K_M`)

Portable head-to-head benchmark for **Qwen 2.5-1.5B-Instruct** on CUDA (**Colab**, **Kaggle**, **Windows**, or **Linux**).
Starts by installing the `v0.2.1` release wheel of `qwen-engine` and `llama-cpp-python` (CUDA).

In [ ]:
import os
import subprocess
import sys

RELEASE_TAG = "v0.2.1"
WHEEL_VER = "0.2.0"
py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
base = f"https://github.com/Areen-09/qwen-engine/releases/download/{RELEASE_TAG}"

if sys.platform == "win32" and (3, 9) <= sys.version_info[:2] <= (3, 13):
    wheel_url = f"{base}/qwen_engine-{WHEEL_VER}-{py_tag}-{py_tag}-win_amd64.whl"
elif sys.platform.startswith("linux") and (3, 9) <= sys.version_info[:2] <= (3, 13):
    wheel_url = f"{base}/qwen_engine-{WHEEL_VER}-{py_tag}-{py_tag}-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl"
else:
    wheel_url = f"{base}/qwen_engine-{WHEEL_VER}.tar.gz"

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", wheel_url, "huggingface_hub"])

env = os.environ.copy()
env["CMAKE_ARGS"] = "-DGGML_CUDA=on"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python"], env=env)

import qwen_engine
print("Installed qwen_engine:", qwen_engine.__version__)

## 1. Benchmark `qwen-engine` (W4A16 Group-128)

In [ ]:
import time
from pathlib import Path
from qwen_engine import Engine

if Path("models/qwen25_1.5b_w4a16.qwenbin").is_file():
    qe = Engine("models/qwen25_1.5b_w4a16.qwenbin", "models/tokenizer.bin", device_id=0)
else:
    qe = Engine.from_pretrained("qwen2.5-1.5b-w4a16", device_id=0)

PROMPT_TOKENS, GEN_TOKENS, RUNS = 128, 128, 5
seed_ids = qe.encode("The quick brown fox jumps over the lazy dog. ")
prompt_text = qe.decode((seed_ids * ((PROMPT_TOKENS // len(seed_ids)) + 1))[:PROMPT_TOKENS])

qe.reset_session()
for _ in qe.stream(prompt_text, max_tokens=16, temperature=0.0):
    pass

qe_pre_list, qe_dec_list = [], []
for _ in range(RUNS):
    qe.reset_session()
    t0 = time.perf_counter()
    it = qe.stream(prompt_text, max_tokens=GEN_TOKENS, temperature=0.0)
    next(it)
    t1 = time.perf_counter()
    n = 1 + sum(1 for _ in it)
    t2 = time.perf_counter()
    qe_pre_list.append(len(qe.encode(prompt_text)) / (t1 - t0))
    qe_dec_list.append((n - 1) / (t2 - t1))

qe_pre = sum(qe_pre_list) / len(qe_pre_list)
qe_dec = sum(qe_dec_list) / len(qe_dec_list)
print(f"qwen-engine -> Prefill: {qe_pre:.2f} tok/s | Decode: {qe_dec:.2f} tok/s")
del qe

## 2. Benchmark `llama.cpp` (`Q4_K_M`)

In [ ]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

local_gguf = Path("models/qwen2.5-1.5b-instruct-q4_k_m.gguf")
gguf_path = str(local_gguf) if local_gguf.is_file() else hf_hub_download(
    repo_id="Qwen/Qwen2.5-1.5B-Instruct-GGUF",
    filename="qwen2.5-1.5b-instruct-q4_k_m.gguf"
)

llm = Llama(model_path=gguf_path, n_gpu_layers=-1, n_ctx=2048, flash_attn=True, verbose=False)
tokens = llm.tokenize(("The quick brown fox jumps over the lazy dog. " * 20).encode("utf-8"))[:PROMPT_TOKENS]
lc_prompt = llm.detokenize(tokens).decode("utf-8", errors="ignore")

llm(lc_prompt, max_tokens=16, temperature=0.0)
lc_pre_list, lc_dec_list = [], []
for _ in range(RUNS):
    t0 = time.perf_counter()
    s = llm(lc_prompt, max_tokens=GEN_TOKENS, temperature=0.0, stream=True)
    next(s)
    t1 = time.perf_counter()
    n = 1 + sum(1 for _ in s)
    t2 = time.perf_counter()
    lc_pre_list.append(len(tokens) / (t1 - t0))
    lc_dec_list.append((n - 1) / (t2 - t1))

lc_pre = sum(lc_pre_list) / len(lc_pre_list)
lc_dec = sum(lc_dec_list) / len(lc_dec_list)

print("\n" + "=" * 60)
print(f"{'Engine':<28} {'Prefill (tok/s)':>15} {'Decode (tok/s)':>15}")
print("-" * 60)
print(f"{'qwen-engine (W4A16 G128)':<28} {qe_pre:>15.2f} {qe_dec:>15.2f}")
print(f"{'llama.cpp (Q4_K_M)':<28} {lc_pre:>15.2f} {lc_dec:>15.2f}")
print("-" * 60)
print(f"Speedup -> Prefill: {qe_pre / lc_pre:.2f}x | Decode: {qe_dec / lc_dec:.2f}x")